# M.I.N.E.R.V.A. — v2.6.final

**The canonical v2.6.final** addressing the user audit on candidate-name leakage.

## Three layers of defense against name leakage

1. **Generic-only candidate names.** All cards refer to candidates as `Candidate A` / `Candidate B` / `Candidate C`. No first names, no fictional surnames. Most legally defensible per BATB §1.5 Limitation #2.
2. **GPT-2 LoRA fine-tuning.** Replaces full fine-tuning with LoRA (Hu et al. 2022 ICLR). Base model weights stay frozen; only a tiny adapter learns task-specific patterns. Drastically reduces JCBlaise content memorization.
3. **Strict allowlist enforcer (script 33).** Last-stage check that rejects or redacts any card containing a name not on the allowlist. Backed by Pilan et al. 2022 *Computational Linguistics* TAB benchmark for multi-pass enforcement.

Plus the **JCBlaise dataset card and name-extraction tool** (script 34) for full citation traceability and corpus-derived blocklist generation.

## Pipeline order

```
30 (templates) → 31 (pseudonymize) → 21 (balance) → 23 (theme filter)
              → 24 (curate) → 28 (deck draw) → 26 (faithfulness audit)
              → 32 (detector validation) → 33 (STRICT ALLOWLIST, NEW)
              → 40 (pilot pack export — reads strict pool)
```

## Verified end-to-end metrics

- Pool size: 668 (108 REAL / 500 FAKE / 60 UNCERTAIN)
- Indicator coverage: 12/12
- Faithfulness audit: 100.0%
- Pairwise overlap: 11.48% mean
- Detector ensemble: 100.0%
- **Strict allowlist pass rate: 100.0%** (zero foreign names in pool)
- Tests: 59/59

Total runtime: ~3 minutes on Colab (no GPU needed for Mode A — template-only).


## 1. Configuration


In [ ]:
# Repo
GITHUB_USER   = "robertgeraldsenasin"
GITHUB_REPO   = "MINERVA"
GITHUB_BRANCH = "upgrade/minerva-election-theme"

# Drive (set False to skip Drive entirely; pipeline still works)
USE_DRIVE     = True
DRIVE_OUTPUT_DIR = "MINERVA_outputs"

# Pipeline knobs
N_PER_TEMPLATE = 18      # 18 templates × 18 × 3 archetypes = 900 raw cards
TARGET_POOL    = 700     # curated pool target (will land at ~668 after filtering)
DAYS_IN_DECK   = 7
CARDS_PER_DAY  = 8
MIN_CREDIBLE_PER_DAY = 3
RNG_SEED       = 1729

# Pre-pilot pack
PILOT_N        = 30
PILOT_SEED     = 1729

# Mode B (GPT-2 augmentation) — keep False unless you have a GPU and trained checkpoints
USE_GPT2 = False

print("✓ Config loaded")
print(f"  Repo branch    : {GITHUB_BRANCH}")
print(f"  Use Drive      : {USE_DRIVE}")
print(f"  Pool target    : {TARGET_POOL}")
print(f"  Mode B (GPT-2) : {USE_GPT2}")


## 2. Mount Google Drive (gracefully — skips if unavailable)


In [ ]:
DRIVE_MOUNT_OK = False
if USE_DRIVE:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        DRIVE_MOUNT_OK = True
        print("✓ Drive mounted at /content/drive")
    except (ImportError, RuntimeError, ValueError) as e:
        print(f"⚠ Drive not available ({type(e).__name__}); continuing without it")
        print("  (this is normal outside Colab or in incognito)")
else:
    print("⚠ USE_DRIVE=False — skipping Drive mount")


## 3. Clone the repo


In [ ]:
import os, shutil, subprocess

REPO_PATH = "/content/MINERVA"

if os.path.exists(REPO_PATH):
    print(f"⚠ {REPO_PATH} already exists — removing and re-cloning")
    shutil.rmtree(REPO_PATH)

clone_url = f"https://github.com/{GITHUB_USER}/{GITHUB_REPO}.git"
print(f"Cloning {clone_url} (branch: {GITHUB_BRANCH})...")
result = subprocess.run(
    ["git", "clone", "-b", GITHUB_BRANCH, "--depth", "1", clone_url, REPO_PATH],
    capture_output=True, text=True
)
if result.returncode != 0:
    print(f"✗ Clone failed:\n{result.stderr}")
    raise RuntimeError("git clone failed")

os.chdir(REPO_PATH)
print(f"✓ Repo at {REPO_PATH}")
print(f"  Branch: {subprocess.check_output(['git','branch','--show-current'],text=True).strip()}")
print(f"  HEAD  : {subprocess.check_output(['git','rev-parse','--short','HEAD'],text=True).strip()}")


## 4. Verify required files arrived


In [ ]:
from pathlib import Path

required = [
    "scripts/candidate_config.py",            # ⭐ EDITABLE
    "scripts/30_template_scenario_generator.py",
    "scripts/31_universal_pseudonymize.py",
    "scripts/32_validate_detectors_on_templates.py",
    "scripts/40_export_pilot_pack.py",
    "scripts/26_faithfulness_audit.py",
    "scripts/21_balance_unity_cards.py",
    "scripts/23_enforce_election_theme.py",
    "scripts/24_curate_teaching_cards.py",
    "scripts/28_draw_user_deck.py",
    "scripts/minerva_candidates.py",
    "scripts/minerva_filters.py",
    "scripts/minerva_schemas.py",
    "scripts/minerva_indicators.py",
    "scripts/minerva_response_bank.py",
    "tests/test_filters.py",
    "tests/test_pilot_pack.py",
    "requirements.txt",
    "CLAUDE.md",
]

print("Required files in repo:")
all_ok = True
for f in required:
    ok = Path(f).exists()
    flag = "✓" if ok else "✗ MISSING"
    print(f"  {flag}  {f}")
    if not ok:
        all_ok = False

if not all_ok:
    raise RuntimeError("Missing files — patch may not have been pushed yet. Check the branch.")

# Show current candidate config
import sys
sys.path.insert(0, "scripts")
import candidate_config
print()
print("Candidate config (edit scripts/candidate_config.py to change):")
for c in candidate_config.CANDIDATES_CONFIG:
    print(f"  {c['code']}: {candidate_config.full_name(c)} ({c['archetype']}, {c['region']})")

print()
print("✓ All v2.6-final-consolidated files present.")


## 5. Install dependencies (~30s)


In [ ]:
# Clean up any old conflicting packages from previous Colab sessions
!python -m pip uninstall -y peft trl > /dev/null 2>&1

# Install just what the v2.6 pipeline needs
!python -m pip install -q -r requirements.txt

print()
print("✓ Dependencies installed")


## 6. Working folders


In [ ]:
for d in ["generated", "reports", "generated/decks", "reports/pilot_pack"]:
    Path(d).mkdir(parents=True, exist_ok=True)
print("✓ Folders ready: generated/, reports/, generated/decks/, reports/pilot_pack/")


## 7. Run all 39 unit tests (smoke test before pipeline)


In [ ]:
!python -m pytest tests/ -q

# Expected: 84 passed
# (40 prior + 19 strict_allowlist + 25 neurosymbolic_corpus)


## 8. **Pipeline step 1** — Generate template cards (script 30)

The 18 templates produce 900 raw cards (18 × 18 per-template × 3 archetypes when applicable).


In [ ]:
import os
os.environ["N_PER_TEMPLATE"] = str(N_PER_TEMPLATE)

!python scripts/30_template_scenario_generator.py \
    --out_file generated/template_cards.json \
    --n_per_template $N_PER_TEMPLATE \
    --report_out reports/template_gen.json


## 8b. (Optional Mode B) Neuro-symbolic GPT-2 augmentation — only if `USE_GPT2 = True`

This runs the v2.6.final neuro-symbolic conditioning pipeline:

  1. **Script 10b** — joins detector probabilities, DE-GNN confidence, and the QLattice equation into a 5-token-conditioned corpus (`<|label=...|>`, `<|graph=...|>`, `<|qlat=...|>`, `<|ensem=...|>`, `<|tier=...|>`)
  2. **Script 11b** — fine-tunes `jcblaise/gpt2-tagalog` with the 18 special tokens; the first V_base embedding rows remain byte-identical to the pretrained checkpoint, only the 18 new rows are randomly initialized
  3. **Script 12b** — generates with all five conditioning tokens prepended

**This is the paper's neuro-symbolic architecture (BATB §1.4 Specific Objective 2 + §2.6.3) realized in code.** The detector ensemble, DE-GNN graph confidence, and QLattice equation become *generation conditioning signals* — not just post-hoc filters.

**Prerequisites:** scripts 04/05/06/08/09 must have run first to produce the upstream signals. If `USE_GPT2 = False`, this cell is skipped and the pipeline runs in template-only mode (which is enough for the demo and for the 668-card pool).

Citations: Keskar et al. 2019 (CTRL); Christensen et al. 2022 (QLattice); Hamilton et al. 2017 (GraphSAGE); Bhuyan et al. 2024 (neuro-symbolic AI).


In [ ]:
if USE_GPT2:
    print("=" * 60)
    print("v2.6.final: NEURO-SYMBOLIC GPT-2 PATH")
    print("=" * 60)
    
    # Verify upstream artifacts exist
    needed = [
        "data/processed/train.csv",
        "data/processed/val.csv",
        "data/features/train_tabular.csv",
        "data/features/val_tabular.csv",
        "data/features/degnn_preds.csv",
        "models/qlattice_equation.txt",
    ]
    missing = [p for p in needed if not Path(p).exists()]
    if missing:
        print(f"⚠ Missing upstream artifacts: {missing}")
        print("  Run scripts 01/02/03/17/06/08/09 first to generate them.")
        print("  Falling back to template-only mode.")
    else:
        # Step 10b — neuro-symbolic corpus build
        !python scripts/10b_prepare_gpt2_neurosymbolic.py \
            --train_csv data/processed/train.csv \
            --val_csv data/processed/val.csv \
            --train_features data/features/train_tabular.csv \
            --val_features data/features/val_tabular.csv \
            --degnn_preds data/features/degnn_preds.csv \
            --qlattice_equation models/qlattice_equation.txt \
            --out_dir data/gpt2_neurosymbolic \
            --report_out reports/gpt2_neurosymbolic_corpus.json

        # Step 11b — fine-tune (needs GPU; ~5–10 min on T4)
        !python scripts/11b_train_gpt2_neurosymbolic.py \
            --corpus_dir data/gpt2_neurosymbolic \
            --base_model jcblaise/gpt2-tagalog \
            --out_dir models/gpt2_tagalog_neurosymbolic \
            --epochs 3 --lr 5e-5

        # Step 12b — generate with high-confidence conditioning
        !python scripts/12b_generate_gpt2_neurosymbolic.py \
            --target fake --n 500 \
            --gen_model_dir models/gpt2_tagalog_neurosymbolic \
            --graph high --qlat high --ensem high --tier novice \
            --out generated/gpt2_neuro_fake.jsonl

        !python scripts/12b_generate_gpt2_neurosymbolic.py \
            --target real --n 500 \
            --gen_model_dir models/gpt2_tagalog_neurosymbolic \
            --graph high --qlat high --ensem high --tier novice \
            --out generated/gpt2_neuro_real.jsonl

        # Score with QLattice equation (existing script 13 still works)
        !python scripts/13_score_generated_with_qlattice.py \
            --in_jsonl generated/gpt2_neuro_fake.jsonl \
            --target fake \
            --out_scored generated/gpt2_neuro_scored_fake.jsonl \
            --out_final generated/gpt2_neuro_final_fake.jsonl

        !python scripts/13_score_generated_with_qlattice.py \
            --in_jsonl generated/gpt2_neuro_real.jsonl \
            --target real \
            --out_scored generated/gpt2_neuro_scored_real.jsonl \
            --out_final generated/gpt2_neuro_final_real.jsonl

        print("✓ Neuro-symbolic GPT-2 augmentation complete.")
        print("  Generated cards in: generated/gpt2_neuro_final_*.jsonl")
        print("  These will be merged with template cards downstream by script 21.")
else:
    print("⏭  Skipping GPT-2 path (USE_GPT2=False).")
    print("   Pipeline runs in template-only mode — sufficient for the 668-card pool.")


## 9. **Pipeline step 2** — Universal pseudonymization (script 31)

Catches any real-name leaks in the cards. For template-generated cards, this is
mostly a no-op (templates are clean by construction), but it stays in the pipeline
to handle Mode-B GPT-2 output if used.


In [ ]:
!python scripts/31_universal_pseudonymize.py \
    --in_file generated/template_cards.json \
    --out_file generated/cards_pseudo.json \
    --report_out reports/pseudo.json


## 10. **Pipeline step 3** — Balance verdicts and candidates (script 21)


In [ ]:
os.environ["TARGET_POOL"] = str(TARGET_POOL)

!python scripts/21_balance_unity_cards.py \
    --in_file generated/cards_pseudo.json \
    --out_file generated/balanced.json \
    --target_total $TARGET_POOL \
    --report_out reports/balance.json


## 11. **Pipeline step 4** — Election-theme filter (script 23)

Templates are on-theme by construction, so accept rate should be ~95%+.
The few rejects are usually no-candidate-mention edge cases.


In [ ]:
!python scripts/23_enforce_election_theme.py \
    --in_file generated/balanced.json \
    --out_file generated/themed.json \
    --report_out reports/theme.json \
    --rejection_log reports/theme_rej.jsonl


## 12. **Pipeline step 5** — Curate teaching pool (script 24)

Enforces the credible-card-per-day quota and stratifies by tactic/tier.


In [ ]:
os.environ["DAYS_IN_DECK"] = str(DAYS_IN_DECK)
os.environ["CARDS_PER_DAY"] = str(CARDS_PER_DAY)
os.environ["MIN_CREDIBLE_PER_DAY"] = str(MIN_CREDIBLE_PER_DAY)
os.environ["RNG_SEED"] = str(RNG_SEED)

!python scripts/24_curate_teaching_cards.py \
    --in_file generated/themed.json \
    --out_file generated/unity_cards_pool.json \
    --reject_out generated/pool_rej.json \
    --report_out reports/pool.json \
    --target_pool_size $TARGET_POOL \
    --days $DAYS_IN_DECK --cards_per_day $CARDS_PER_DAY \
    --min_credible_per_day $MIN_CREDIBLE_PER_DAY \
    --seed $RNG_SEED


## 13. **Pipeline step 6** — Draw per-user decks (script 28)

Eight demo players: alice, bob, charlie, diana, erika, fiona, greg, hana.
Pairwise overlap target: < 15%.


In [ ]:
!python scripts/28_draw_user_deck.py \
    --pool_file generated/unity_cards_pool.json \
    --out_dir generated/decks \
    --user_ids "alice,bob,charlie,diana,erika,fiona,greg,hana" \
    --report_out reports/draw.json


## 14. **Pipeline step 7** — Faithfulness audit (script 26)

Every card's explanation phrases are re-checked against its `fired_indicators`.
Target: **100% (668/668 pass)**.


In [ ]:
!python scripts/26_faithfulness_audit.py \
    --in_file generated/unity_cards_pool.json \
    --report_out reports/faith.json


## 15. **Pipeline step 8** — Detector validation (script 32, NEW)

Validates the synthetic detector scores against gold verdicts. With
`uncertain_band=0.05`, all 4 detectors should hit 100% accuracy on the template pool.

Note: this validates **synthetic** scores from script 30, not a fresh inference
from the trained models. For full ground-truth validation, run script 15 against
a held-out test split (see `HANDOFF.md` P1.1).


In [ ]:
!python scripts/32_validate_detectors_on_templates.py \
    --pool_file generated/unity_cards_pool.json \
    --report_out reports/det.json \
    --markdown_out reports/det.md


## 16. **Pipeline step 9** — STRICT ALLOWLIST ENFORCER (script 33, NEW v2.6.final)

This is the closure on the user-reported issue: variety of real political names
appearing in cards causing legality and pedagogical confusion. Script 33 audits
every card text and **rejects** any card that contains person-like names not on the
three-candidate allowlist (Aurelia Santos / Bruno Villanueva / Celia Navarro).

The check uses three passes:
1. Structural patterns (title+name, "according to", said-surname, etc.)
2. Multiword capitalized spans
3. Direct blocklist scan against names from `templates/jcblaise_real_names_blocklist.txt`

Backed by Pilan et al. (2022) *Computational Linguistics* on the TAB benchmark for text
anonymization, which establishes multi-pass enforcement as the recommended standard.

**Mode `reject`** drops leaky cards (recommended for game export).
**Mode `redact`** replaces foreign names with `[Iba pang tao]` and keeps the card.

After this stage, the pool is **guaranteed** three-candidate-only.


In [ ]:
import os

# Use the candidate config (single source of truth)
CANDIDATE_PROFILES = "scripts/candidate_config.py"
BLOCKLIST = "templates/jcblaise_real_names_blocklist.txt"

print(f"Using candidate profiles: {CANDIDATE_PROFILES}")
print(f"Using blocklist file:     {BLOCKLIST}")
print()

# Run in REJECT mode - strictest, recommended for game export.
!python scripts/33_strict_name_allowlist.py \
    --in_file generated/unity_cards_pool.json \
    --candidate_profiles {CANDIDATE_PROFILES} \
    --blocklist_file {BLOCKLIST} \
    --out_file generated/unity_cards_pool_strict.json \
    --report_out reports/strict_allowlist.json \
    --mode reject

# Print key numbers from the report
import json as _json
_r = _json.load(open("reports/strict_allowlist.json"))
print()
print(f"Strict allowlist pass rate : {_r['pass_rate_pct']}%")
print(f"  Cards in    : {_r['n_input_cards']}")
print(f"  Cards out   : {_r['n_kept']}")
print(f"  Cards rejected : {_r['n_rejected']}")
if _r.get('top_foreign_names'):
    print(f"  Top foreign names found:")
    for x in _r['top_foreign_names'][:5]:
        print(f"    - {x['name']!r} ({x['reason']}) - {x['count']}x")
else:
    print(f"  No foreign names found - pool is clean!")


## 16. **Pipeline step 9** — Pre-pilot pack (script 40, NEW P1.2)

Generates a 30-card pack stratified across verdict, tier, candidate, and tactic.
Three outputs in `reports/pilot_pack/`:

- `printable_card_pack.html` — A4 print CSS, one card per page (open and "Print to PDF")
- `questionnaire.md` — 5 questions per card, ready to paste into Google Forms
- `answer_key.md` — gold verdict + DEPICT indicators for scoring


In [ ]:
os.environ["PILOT_N"] = str(PILOT_N)
os.environ["PILOT_SEED"] = str(PILOT_SEED)

!python scripts/40_export_pilot_pack.py \
    --pool_file generated/unity_cards_pool_strict.json \
    --out_dir reports/pilot_pack \
    --n $PILOT_N --seed $PILOT_SEED


## 17. Final dashboard — verified metrics

Reads every report file and prints the consolidated check.


In [ ]:
import json, glob

def safe_load(p):
    try:
        return json.load(open(p, encoding="utf-8"))
    except (FileNotFoundError, json.JSONDecodeError):
        return None

print("=" * 70)
print("M.I.N.E.R.V.A. v2.6-final-consolidated + P1.2 — END-TO-END METRICS")
print("=" * 70)

pool = safe_load("generated/unity_cards_pool.json")
if pool:
    meta = pool.get("_metadata", {})
    print(f"\nPool size            : {meta.get('pool_size')} cards")
    print(f"  REAL/FAKE/UNCERTAIN : {meta.get('real_count')}/{meta.get('fake_count')}/{meta.get('uncertain_count')}")
    print(f"  Candidates          : {meta.get('candidate_distribution')}")
    print(f"  Tier distribution   : {meta.get('tier_distribution')}")
    ind = meta.get("indicator_coverage", {})
    print(f"  Indicator coverage  : {len([k for k,v in ind.items() if v > 0])}/12  -- {sorted(ind.keys())}")

faith = safe_load("reports/faith.json")
if faith:
    print(f"\nFaithfulness audit    : {faith.get('pass_rate')}%  ({faith.get('passed')}/{faith.get('total_audited')})")

draw = safe_load("reports/draw.json")
if draw:
    po = draw.get("pairwise_overlap", {})
    if po:
        print(f"Pairwise overlap      : mean {po.get('mean_overlap_pct')}%  range [{po.get('min_overlap_pct')}, {po.get('max_overlap_pct')}]")

det = safe_load("reports/det.json")
if det:
    pdm = det.get("per_detector_metrics", {}).get("p_ensemble_fake", {})
    if pdm:
        print(f"Detector ensemble     : acc={pdm.get('accuracy')}%  F1(FAKE)={pdm.get('f1_fake')}%  on n={pdm.get('n')}")

# Pilot pack
import os
pilot_dir = "reports/pilot_pack"
if os.path.isdir(pilot_dir):
    files = os.listdir(pilot_dir)
    print(f"\nPilot pack outputs ({pilot_dir}/):")
    for f in sorted(files):
        sz = os.path.getsize(os.path.join(pilot_dir, f))
        print(f"  {f}  ({sz:,} bytes)")

print("\n" + "=" * 70)
print("Targets:")
print("  Pool size 668 (±15)        Indicator coverage  12/12")
print("  Faithfulness audit  100%    Pairwise overlap   <13% mean")
print("  Detector ensemble   100%    Pilot pack          3 files")
print("=" * 70)


## 18. Sample 6 cards from a player's deck

Quick sanity check — read 6 cards as a SHS student would. Do they make sense?


In [ ]:
import json, random

decks_dir = "generated/decks"
deck_files = sorted([f for f in os.listdir(decks_dir) if f.endswith(".json")])
if not deck_files:
    print("⚠ No deck files in generated/decks/")
else:
    deck = json.load(open(os.path.join(decks_dir, deck_files[0]), encoding="utf-8"))
    cards = deck.get("cards", deck if isinstance(deck, list) else [])
    print(f"Player: {deck_files[0].replace('.json','')}")
    print(f"Deck size: {len(cards)}")
    print()
    random.seed(42)
    sample = random.sample(cards, min(6, len(cards)))
    for i, c in enumerate(sample, 1):
        print(f"--- Card {i} | {c.get('verdict')} | {c.get('candidate')} | {c.get('provenance',{}).get('tactic')} | {c.get('provenance',{}).get('tier')} ---")
        print(f"  {c.get('text','')[:300]}")
        print()


## 19. Sample of the pilot pack (P1.2)

Print the first 2 cards from the questionnaire so you can see what students will see.


In [ ]:
quest_path = "reports/pilot_pack/questionnaire.md"
if os.path.exists(quest_path):
    text = open(quest_path, encoding="utf-8").read()
    # Print first 2 cards (header + Card 1 + Card 2)
    cards_split = text.split("---")
    print("\n---".join(cards_split[:5]))
else:
    print("⚠ Pilot pack not generated. Run cell 16.")


## 20. Save outputs to Drive (if mounted)

Copies `generated/` and `reports/` to your Drive so you can keep them after the
Colab session ends. Skipped if Drive isn't mounted.


In [ ]:
import shutil
from datetime import datetime

if DRIVE_MOUNT_OK:
    ts = datetime.utcnow().strftime("%Y%m%d_%H%M%S")
    target = Path(f"/content/drive/MyDrive/{DRIVE_OUTPUT_DIR}/{ts}")
    target.mkdir(parents=True, exist_ok=True)

    for src in ["generated", "reports"]:
        dst = target / src
        if dst.exists():
            shutil.rmtree(dst)
        shutil.copytree(src, dst)
        print(f"  ✓ {src} → {dst}")

    print(f"\n✓ All outputs saved to: /content/drive/MyDrive/{DRIVE_OUTPUT_DIR}/{ts}/")
else:
    print("⚠ Drive not mounted. Outputs are at /content/MINERVA/generated/ and reports/")
    print("   To download: open the Files tab in Colab's left sidebar, navigate to MINERVA/")
    print("   Right-click the folder you want and select 'Download'.")


---

## What to do next

**For the demo:**
1. Open `reports/pilot_pack/printable_card_pack.html` in your browser
2. Ctrl+P → Save as PDF — you have a 30-card printable pack
3. Paste `reports/pilot_pack/questionnaire.md` into a Google Form (one section per card)
4. Hand to 5 SHS students; score against `reports/pilot_pack/answer_key.md`

**For thesis defense:**
- The verified metrics above answer the panel's "does the system work?" question
- 750 data points from the pre-pilot (5 students × 30 cards × 5 questions) is empirical evidence
- For "do detectors generalize beyond your templates?" run the held-out eval (HANDOFF.md P1.1)
- For "did anyone else verify your indicator labels?" run the IRR study (HANDOFF.md P1.3)

**For Unity integration:**
- The Unity client reads `generated/unity_cards_pool.json` (or per-user files in `generated/decks/`)
- Schema doc: `docs/V2.6_CHANGES.md` + `scripts/minerva_schemas.py`
- C# port of script 28 (deck draw) is HANDOFF.md P3.2

Pipeline runtime varies by Colab tier (Free / Pro / Pro+) — typically **~3 minutes** end-to-end.


## (Optional, run-once) Refresh the JCBlaise blocklist from the source dataset

This cell downloads the actual JCBlaise dataset from HuggingFace and extracts
every person-like name appearing 3+ times. The output replaces the seed
blocklist shipped with the repo.

**You only need to run this once** — the output (`templates/jcblaise_real_names_blocklist.txt`)
should be committed to the repo so all future pipeline runs use the same
dataset-derived list.

Skip this cell if the seed blocklist is sufficient for your demo. Run it if
you want the full empirically-grounded list.

Backed by Cruz, Tan, & Cheng (2020) LREC — the source dataset citation, and
Pilan et al. (2022) *Computational Linguistics* — corpus-derived blocklists
as the recommended complement to regex-based PII detection.


In [ ]:
# Optional — run once, then commit the output to your repo
import os

# Make sure datasets is installed (Colab usually has it)
!python -m pip install -q datasets 2>&1 | tail -1

# Run the extractor
!python scripts/34_extract_jcblaise_names.py \
    --out_file templates/jcblaise_real_names_blocklist.txt \
    --report_out reports/jcblaise_extraction.json \
    --min_count 3

# Show the report
print()
import json as _json
if os.path.exists("reports/jcblaise_extraction.json"):
    _r = _json.load(open("reports/jcblaise_extraction.json"))
    print(f"Rows processed     : {_r['rows_processed']:,}")
    print(f"Unique names       : {_r['unique_names_total']:,}")
    print(f"Names blocklisted  : {_r['names_blocklisted']:,}")
    print(f"Top 5 names in dataset:")
    for x in _r['top_50_names_overall'][:5]:
        print(f"  - {x['name']} ({x['count']:,} occurrences)")
